Just get a lot of GenZ lingo. Nothing special. these are hand done genz_00 - genz_05
- https://huggingface.co/datasets/MLBtrio/genz-slang-dataset
- https://huggingface.co/datasets/GCruz19/GenZ_data
- https://huggingface.co/datasets/Programmer-RD-AI/genz-slang-pairs-1k
- https://huggingface.co/datasets/synk/genz-slang-completions
- https://huggingface.co/datasets/aiml8726737/gen-alpha-brainrot
- https://huggingface.co/datasets/AliceYin/genz-contextual-abusive-slang-v2-silver

In [6]:
import tiktoken
import numpy as np
from datasets import load_dataset
import json

In [88]:
import os

file = f"genz_00_train.bin"
n = os.path.getsize(file) // 2
print(f"{file}: {n:,} tokens, {n/1e6:.2f}M tokens")

genz_00_train.bin: 421,753 tokens, 0.42M tokens


Ok I should redistribute and split 01 into many others but I also want less of this "abusive" dataset

___

In [78]:
enc = tiktoken.get_encoding("gpt2")
EOT = enc.eot_token


In [79]:
ds = load_dataset("aiml8726737/gen-alpha-brainrot", num_proc=8)

ids = []
for row in ds["train"]:
	# print(row["messages"][1]["content"])
	genz_slang = row["messages"][1]["content"]
	curr_ids = enc.encode_ordinary(genz_slang) + [EOT]
	ids.append(curr_ids)

___

In [80]:
ds = load_dataset("AliceYin/genz-contextual-abusive-slang-v2-silver", "full",  num_proc=8)

In [81]:
for row in ds["full"]:
	slang = row["text"]
	curr_ids = enc.encode_ordinary(slang) + [EOT]
	ids.append(curr_ids)

___

In [82]:
ds = load_dataset("synk/genz-slang-completions",  num_proc=8)

for row in ds["train"]:
	slang = row["completion"]
	curr_ids = enc.encode_ordinary(slang) + [EOT]
	ids.append(curr_ids)

___

In [83]:
ds = load_dataset("Programmer-RD-AI/genz-slang-pairs-1k",  num_proc=8);

for row in ds["train"]:
	slang = row["gen_z"]
	curr_ids = enc.encode_ordinary(slang) + [EOT]
	ids.append(curr_ids)

___

In [84]:

ds = load_dataset("GCruz19/GenZ_data",  num_proc=8);

for row in ds["train"]:
	try:
		slang = json.loads(row['translation'])['genz']
		curr_ids = enc.encode_ordinary(slang) + [EOT]
		ids.append(curr_ids)
	except Exception as e:
		print(f"Error processing row: {row}. Error: {e}")

Error processing row: {'id': '820', 'translation': '{ "normal": "His dance moves are impressive; he should be on TikTok.", "genz": "His dance moves are fire, he should be on TikTok\n" }'}. Error: Invalid control character at: line 1 column 129 (char 128)


___

In [85]:
ds = load_dataset("MLBtrio/genz-slang-dataset",  num_proc=8);

for row in ds["train"]:
	slang1 = row['Slang']
	slang2 = row['Example']
	curr_ids = enc.encode_ordinary(slang1) + [EOT]
	ids.append(curr_ids)
	curr_ids = enc.encode_ordinary(slang2) + [EOT]
	ids.append(curr_ids)

___

In [86]:
len(ids)

17762

In [89]:
arr = np.array(ids, dtype=object)
ixs = np.random.permutation(len(arr))
flat = np.concatenate(arr[ixs]).astype(np.uint16)
print(len(flat))
flat.tofile("genz_00_train.bin")

421753


___
# Big D

In [3]:
from huggingface_hub import list_repo_files
files = [f for f in list_repo_files("tiagozip/tiktok", repo_type="dataset")
	if f.endswith(".parquet") and "comments" in f]
files

['data/comments/2018-01.parquet',
 'data/comments/2018-02.parquet',
 'data/comments/2018-03.parquet',
 'data/comments/2018-04.parquet',
 'data/comments/2018-05.parquet',
 'data/comments/2018-06.parquet',
 'data/comments/2018-07.parquet',
 'data/comments/2018-08.parquet',
 'data/comments/2018-09.parquet',
 'data/comments/2018-10.parquet',
 'data/comments/2018-11.parquet',
 'data/comments/2018-12.parquet',
 'data/comments/2019-01.parquet',
 'data/comments/2019-02.parquet',
 'data/comments/2019-03.parquet',
 'data/comments/2019-04.parquet',
 'data/comments/2019-05.parquet',
 'data/comments/2019-06.parquet',
 'data/comments/2019-07.parquet',
 'data/comments/2019-08.parquet',
 'data/comments/2019-09.parquet',
 'data/comments/2019-10.parquet',
 'data/comments/2019-11.parquet',
 'data/comments/2019-12.parquet',
 'data/comments/2020-01.parquet',
 'data/comments/2020-02.parquet',
 'data/comments/2020-03.parquet',
 'data/comments/2020-04.parquet',
 'data/comments/2020-05.parquet',
 'data/comment

In [8]:
gpt2 = tiktoken.get_encoding("gpt2")
o200k = tiktoken.get_encoding("o200k_base")

for w in ["rizz", "delulu", "skibidi", "no cap", "💀"]:
    print(w, len(gpt2.encode(w)), len(o200k.encode(w)))

rizz 2 2
delulu 2 2
skibidi 3 3
no cap 2 2
💀 3 2


In [ ]:
import modal
vol = modal.Volume.from_name("genz")

files = [f"databig/bigdata_{i}.bin" for i in range(3, 32)]
for p in files:
    vol.remove_file(p)

InvalidError: No such file or directory.